# VS Code Remote Notebook Validation

Sign in at your deployment's `/workspaces/connections` page, select your running workspace and requested duration, generate a connection, and copy its URL. In VS Code choose **Select Kernel > Select Another Kernel > Existing Jupyter Server**, paste the generated desktop URL including its token, and select the remote Python kernel. Do not enter Google passwords, OAuth client secrets, or browser cookies. See [the customer guide](USER_GUIDE.md#vs-code-jupyter-extension).

Run Cell 2 and compare its hostname with the actual Workspace pod. On 2026-09-12, the standard Microsoft Jupyter extension executed this cell against the pilot's token-authenticated public endpoint: hostname `ws-gke-pilot-2qdwp-0`, interpreter `/opt/conda/bin/python`, and result `42`. The deployment also passed a separate public kernel-WebSocket test and closed its active connection on grant revocation. These are scoped checks, not production security certification.

Connection duration is administrator-configurable: 24 hours by default and a seven-day requested maximum. The tested GKE cluster shortened requests above two days to 48 hours. Always check the displayed expiry. Generate a replacement URL before expiry, select the existing kernel when available, and use **Revoke** for the old grant once the replacement works. Token expiry does not stop the workspace or explicitly shut down the kernel; culling and other lifecycle policies are separate. A public HTTP/WebSocket test preserved a variable in the same kernel across credential replacement, but automatic VS Code renewal and multi-day operation remain unverified.

Verify Run All, save/reopen, kernel restart, interruption, and existing-kernel selection after token replacement separately when testing a new customer environment.

This notebook is stored in the VS Code workspace. Files read or written by Python live on the remote kernel filesystem. Never store a connection token or token URL in notebook metadata, source, or outputs.


In [ ]:
"""Verify a GKE remote kernel without displaying credentials."""

import os
import platform
import sys
from pathlib import Path

hostname = platform.node()
print({"hostname": hostname, "platform": platform.system(), "interpreter": sys.executable})
assert platform.system() == "Linux", "Select the GKE notebook kernel, not local Python"
assert hostname.startswith("ws-"), "Compare this hostname with the actual workspace pod"
assert Path("/home/jovyan").is_dir(), "Expected the notebook home directory"
assert os.getuid() == 1000, "Expected the unprivileged notebook user"
assert not Path("/var/run/secrets/kubernetes.io/serviceaccount/token").exists()
assert 6 * 7 == 42
print("Remote notebook kernel validation passed:", 6 * 7)
